# AR Rollout Loss — Sanity Checks

Two checks:
1. **Rollout error curve** — L1 error vs prediction step index for TF vs AR.
   A well-trained AR loss should show slower error growth than TF-only.
3. **Feature distribution overlap** — PCA of predicted vs true patch embeddings.
   AR predictions should cluster closer to the true distribution than TF.

Requires a checkpoint trained with `ar_steps: 1` and a small validation batch.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from einops import repeat
from sklearn.decomposition import PCA

from rmind.components.base import Modality
from rmind.components.transformer.decoder import CrossAttentionDecoderHead
from rmind.models.control_transformer import ControlTransformer

# ── user config ────────────────────────────────────────────────────────────────
CHECKPOINT = Path("../checkpoints/ar_rollout/last.ckpt")  # adjust to your path
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_BATCHES = 4
# ───────────────────────────────────────────────────────────────────────────────

## Load model + datamodule

In [ ]:
import sys

from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

model = ControlTransformer.load_from_checkpoint(CHECKPOINT, map_location=DEVICE)
model.eval()

sys.path.insert(0, "..")
with initialize_config_dir(
    config_dir=str(Path("../config").resolve()), version_base=None
):
    cfg = compose(
        config_name="train",
        overrides=[
            "+experiment=yaak/control_transformer/pretrain_vjepa",
            "data.num_workers=0",
            "data.batch_size=2",
        ],
    )

datamodule = instantiate(cfg.data)
datamodule.setup("validate")
val_loader = datamodule.val_dataloader()

## Collect TF and AR predictions

In [ ]:
from typing import TYPE_CHECKING, Any, cast

if TYPE_CHECKING:
    from rmind.components.objectives.forward_dynamics import (
        ForwardDynamicsPredictionObjective,
    )

all_tf_preds: list[torch.Tensor] = []  # [B, T-1, N, D]
all_ar_preds: list[torch.Tensor] = []  # [B, T-2, N, D]
all_targets: list[torch.Tensor] = []  # [B, T-1, N, D]

objective = cast(
    "ForwardDynamicsPredictionObjective", model.objectives["forward_dynamics"]
)
foresight_cam = ("foresight", "cam_front_left")

with torch.no_grad():
    for i, raw_batch in enumerate(val_loader):
        if i >= N_BATCHES:
            break

        batch = {
            k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
            for k, v in raw_batch.items()
        }
        episode = model.episode_builder(batch)
        embedding = model.encoder(
            src=episode.embeddings_unpacked, mask=episode.attention_mask
        )

        metrics: Any = objective.compute_metrics(episode=episode, embedding=embedding)

        tf_logits: torch.Tensor = metrics["_artifacts"]["last_embeddings"][
            Modality.FORESIGHT
        ]["cam_front_left"]
        targets: torch.Tensor = metrics["_artifacts"]["last_targets"][
            Modality.FORESIGHT
        ]["cam_front_left"]

        all_tf_preds.append(tf_logits.cpu())
        all_targets.append(targets.cpu())

        if objective.ar_steps > 0 and objective.feedback_projection is not None:
            index = episode.index[:-1]
            k = (Modality.SUMMARY, "action_summary")
            action_summary = index.select(k).parse(embedding).get(k)

            _, _, n_patches, _ = episode.embeddings.get((
                Modality.IMAGE,
                "cam_front_left",
            )).shape
            mask_tokens = repeat(
                episode.embeddings.get((Modality.UTILITY, "mask"))[:, 1:],
                "b t 1 d -> b t n d",
                n=n_patches,
            )
            if objective.patch_pos_embed is not None:
                mask_tokens = objective.patch_pos_embed(mask_tokens)

            z_hat_proj = objective.feedback_projection(tf_logits)
            ar_context = torch.cat([z_hat_proj[:, :-1], action_summary[:, 1:]], dim=-2)
            ar_logits = objective.heads.get(foresight_cam)(
                CrossAttentionDecoderHead.Input(
                    query=mask_tokens[:, :-1], context=ar_context
                )
            )
            all_ar_preds.append(ar_logits.cpu())

## Sanity Check 1 — Rollout Error Curve

L1 error per prediction step: TF predicts `z[t+1]` from observed `z[t]`; AR predicts `z[t+2]` from its own prediction `z_hat[t+1]`. Both are conditioned on the true action.

We expect:
- TF step-1 error is the training loss baseline.
- AR step-2 error should be lower (or at least not diverge badly) compared to a model trained without the AR loss.

In [ ]:
tf_stack = torch.cat(all_tf_preds, dim=0)  # [N_total, T-1, n_patches, D]
tgt_stack = torch.cat(all_targets, dim=0)  # [N_total, T-1, n_patches, D]

# TF error per timestep slot (mean over batch, patches, dim)
tf_err_per_step = (tf_stack - tgt_stack).abs().mean(dim=(0, 2, 3))  # [T-1]

fig, ax = plt.subplots(figsize=(7, 4))
steps = np.arange(1, len(tf_err_per_step) + 1)
ax.plot(steps, tf_err_per_step.numpy(), marker="o", label="TF (step 1 ... T-1)")

if all_ar_preds:
    ar_stack = torch.cat(all_ar_preds, dim=0)  # [N_total, T-2, n_patches, D]
    ar_err_per_step = (ar_stack - tgt_stack[:, 1:]).abs().mean(dim=(0, 2, 3))  # [T-2]
    ar_steps_x = np.arange(2, len(ar_err_per_step) + 2)
    ax.plot(
        ar_steps_x,
        ar_err_per_step.numpy(),
        marker="s",
        linestyle="--",
        label="AR (step 2 ... T-1)",
    )

ax.set_xlabel("Prediction step (1 = next frame)")
ax.set_ylabel("Mean L1 error")
ax.set_title("Rollout error curve — TF vs AR")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("rollout_error_curve.png", dpi=150)
plt.show()

**What to look for:**
- The AR curve at step 2 should be *below* what you'd get if you just naively extended the TF step-1 error (i.e. below the TF value at step 2 from a TF-only model).
- If AR error is *much higher* than TF at the same step, the feedback signal is not helping — revisit `feedback_projection` capacity or learning rate.

## Sanity Check 3 — Feature Distribution Overlap

PCA of predicted patch embeddings vs ground-truth patch embeddings.  
Good AR predictions should overlap with the true distribution rather than drifting.

We use the first 2 principal components computed on the true embeddings.

In [ ]:
def flatten_step(tensor: torch.Tensor, step_idx: int = 0) -> np.ndarray:
    """Extract patches at a given timestep: [B, T, N, D] -> [(B*N), D]."""
    x = tensor[:, step_idx]  # [B, N, D]
    return x.reshape(-1, x.shape[-1]).numpy()


true_feats = flatten_step(tgt_stack, step_idx=0)
tf_feats = flatten_step(tf_stack, step_idx=0)

pca = PCA(n_components=2)
pca.fit(true_feats)

true_2d = pca.transform(true_feats)
tf_2d = pca.transform(tf_feats)

n_plots = 2 if all_ar_preds else 1
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]


def scatter_pair(ax: plt.Axes, pred_2d: np.ndarray, label: str, title: str) -> None:
    ax.scatter(*true_2d.T, s=4, alpha=0.3, label="true z", color="steelblue")
    ax.scatter(*pred_2d.T, s=4, alpha=0.3, label=label, color="tomato")
    ax.set_title(title)
    ax.legend(markerscale=3)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")


scatter_pair(axes[0], tf_2d, "TF pred", "TF: predicted vs true (step 1)")

if all_ar_preds:
    ar_stack = torch.cat(all_ar_preds, dim=0)
    ar_feats = flatten_step(ar_stack, step_idx=0)
    true_step2_feats = flatten_step(tgt_stack, step_idx=1)
    true_step2_2d = pca.transform(true_step2_feats)
    ar_2d = pca.transform(ar_feats)

    ax2 = axes[1]
    ax2.scatter(*true_step2_2d.T, s=4, alpha=0.3, label="true z[2]", color="steelblue")
    ax2.scatter(*ar_2d.T, s=4, alpha=0.3, label="AR pred z[2]", color="tomato")
    ax2.set_title("AR: predicted vs true (step 2)")
    ax2.legend(markerscale=3)
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")

plt.suptitle("Feature distribution overlap — PCA of patch embeddings", y=1.01)
plt.tight_layout()
plt.savefig("feature_distribution.png", dpi=150)
plt.show()

**What to look for:**
- TF and AR predicted distributions should visually overlap with the true distribution (not form a separate cluster).
- If predictions form a tight cluster far from the true distribution, the model is predicting a constant or collapsed mode — check the loss weight and whether the `feedback_projection` has been initialized properly.
- PCA explains variance; for a quantitative score use average cosine similarity between predicted and nearest true embedding.

## Bonus: mean cosine similarity

In [ ]:
true_t = tgt_stack[:, 0].reshape(-1, tgt_stack.shape[-1])
tf_t = tf_stack[:, 0].reshape(-1, tf_stack.shape[-1])

cos_tf = F.cosine_similarity(tf_t, true_t, dim=-1).mean().item()
print(f"TF  step-1 mean cosine sim: {cos_tf:.4f}")  # noqa: T201

if all_ar_preds:
    true_t2 = tgt_stack[:, 1].reshape(-1, tgt_stack.shape[-1])
    ar_t2 = torch.cat(all_ar_preds, dim=0)[:, 0].reshape(-1, tgt_stack.shape[-1])
    cos_ar = F.cosine_similarity(ar_t2, true_t2, dim=-1).mean().item()
    print(f"AR  step-2 mean cosine sim: {cos_ar:.4f}")  # noqa: T201